# 04 — Model B: G0 gloss → motion (3 seeds)

**Purpose.** Train the main motion model on G0 pseudo-gloss input with real epochs and early stopping on VAL; dry run and resume test first.

**Kaggle settings.** Accelerator **GPU** · Internet On (not strictly needed).

**Inputs.** Dataset `bdsl-k400` + **Add Input → Output of notebook 03**.

**Outputs** (under `/kaggle/working/k400/`, saved by *Save Version → Save & Run All*). `outputs/runs/gloss2motion_gloss_kaggle400_<time>/` (run.json, env.json, train.log, metrics.json, per_item.csv, seed*/curves.csv, checkpoints), `outputs/registry.csv`.

**Expected run time** [ESTIMATE, not measured yet]. 20–60 minutes on a GPU for 3 seeds + about 5 minutes for the dry run and resume test.

**What to check.** Dry run: `device cuda`, finite loss. Training: 3 seeds finished, curves fall, best epochs printed. Compare Model B with B0/B1 on VAL — if Model B is not better, that is a result, report it.

**How training works.** Model B (non-autoregressive Transformer, 2.7 M parameters on the pilot vocabulary) learns
G0 gloss tokens → 208 motion values per frame. Max 600 epochs, early stopping on **VAL** masked position L1
(patience 60), seeds 42 / 43 / 44, AdamW 5e-4. Training reads only train + val; reading test raises an error.
If the session stops: *Save Version*, then a new version with this Output as Input and run
`S.bdsl("train-gloss2motion", "--data", "kaggle400", "--input", INPUT, "--resume", "<run folder>")`.
> Honesty rules: label every number measured or estimated; results are small-scale and signer-dependent;
> the test split is used once (notebook 07); low coordinate error is not proof of correct sign language;
> G0 glosses are pseudo-glosses, not BdSL annotation; no human or expert validation exists.

In [ ]:
# Cell 1 - Setup (the same in every notebook). Finds the data and the code, sets the paths, and copies the
# outputs of earlier notebooks (added with "Add Input") into /kaggle/working/k400 so results accumulate.
import os, sys, zipfile, pathlib, json
NOTEBOOK = "04_train_model_b"

def _find_root():
    base = pathlib.Path("/kaggle/input")
    if base.is_dir():
        for dirpath, dirnames, filenames in os.walk(base):
            if "UPLOAD_MANIFEST.csv" in filenames and "PACKAGE_INFO.json" in filenames:
                return pathlib.Path(dirpath)
            depth = len(pathlib.Path(dirpath).relative_to(base).parts)
            dirnames[:] = [] if depth >= 4 else [d for d in dirnames if d not in ("videos", "k400")]
        zips = sorted(base.rglob("bdsl-k400-*.zip"))
        if zips:  # Kaggle kept the upload as one zip file: unpack it once (a few minutes)
            dest = pathlib.Path("/tmp/k400_dataset")   # /tmp is not saved as Output
            if not (dest / "UPLOAD_MANIFEST.csv").is_file():
                print("Unpacking", zips[0], "->", dest)
                zipfile.ZipFile(zips[0]).extractall(dest)
            return dest
        raise FileNotFoundError("The private dataset is not attached. Right panel: Add Input -> Your Datasets -> bdsl-k400.")
    here = pathlib.Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "pyproject.toml").is_file() and (c / "kaggle" / "src" / "k400").is_dir():
            return c
    raise FileNotFoundError("Open this notebook inside the project folder, or on Kaggle with the dataset attached.")

DATASET_ROOT = _find_root()
CODE_ROOT = DATASET_ROOT / "code" if (DATASET_ROOT / "code" / "pyproject.toml").is_file() else DATASET_ROOT
for _p in (CODE_ROOT / "kaggle" / "src", CODE_ROOT / "src"):
    sys.path.insert(0, str(_p))
from k400.setup import setup
S = setup(DATASET_ROOT, CODE_ROOT, NOTEBOOK)

In [ ]:
INPUT = "gloss"
B = ["train-gloss2motion", "--data", "kaggle400", "--input", INPUT]

## Step 1 — Dry run: one optimiser step on the GPU (never a result, named DRYRUN_)

In [ ]:
rc_dry = S.bdsl(*B, "--max-steps", "1")
import glob
dry = sorted(glob.glob(str(S.state_root / f"outputs/runs/DRYRUN_gloss2motion_{INPUT}_*")))[-1]
dry_meta = json.load(open(pathlib.Path(dry) / "run.json"))
print("device:", dry_meta["device"], "|", dry_meta["device_note"])
print(dry_meta["dry_run"])

## Step 2 — Resume test (engineering check, not for the paper): 4 epochs straight vs 2 + resume 2

In [ ]:
R = [*B, "--not-for-paper", "--set", "gloss2motion.train.max_epochs=4", "--set", "gloss2motion.seeds=[42]"]
S.bdsl(*R)
straight = sorted(glob.glob(str(S.state_root / "outputs/runs/NFP_gloss2motion_gloss_*")))[-1]
S.bdsl(*R, "--stop-after-epochs", "2")
stopped = sorted(glob.glob(str(S.state_root / "outputs/runs/NFP_gloss2motion_gloss_*")))[-1]
S.bdsl(*R, "--resume", stopped)
import torch
wa = torch.load(pathlib.Path(straight) / "seed42/ckpt_last.pt", map_location="cpu", weights_only=True)["model"]
wb = torch.load(pathlib.Path(stopped) / "seed42/ckpt_last.pt", map_location="cpu", weights_only=True)["model"]
max_diff = max(float((wa[k] - wb[k]).abs().max()) for k in wa)
print("max |weight difference| straight vs resumed:", max_diff, "(0 = identical; values far below 1e-3 can come from GPU non-determinism; a resume bug gives large differences)")

## Step 3 — Real training: 3 seeds, early stopping on VAL

In [ ]:
rc_train = S.bdsl(*B)
run_dir = pathlib.Path(sorted(d for d in glob.glob(str(S.state_root / f"outputs/runs/gloss2motion_{INPUT}_kaggle400_*")) if (pathlib.Path(d) / "metrics.json").is_file())[-1])
print("run:", run_dir.name)

## Step 4 — Curves (train vs val) and VAL comparison with B0 / B1 (measured)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
mets = json.load(open(run_dir / "metrics.json"))
meta = json.load(open(run_dir / "run.json"))
env = json.load(open(run_dir / "env.json"))
print("GPU:", env["device"]["gpu_names"], "| device:", meta["device"], "| seeds:", meta["seeds"])
fig, ax = plt.subplots(1, len(meta["seeds"]), figsize=(5 * len(meta["seeds"]), 3.2))
for a, s in zip(ax, meta["seeds"]):
    c = pd.read_csv(run_dir / f"seed{s}/curves.csv")
    a.plot(c["epoch"], c["train_pos"], label="train (pos L1)"); a.plot(c["epoch"], c["monitor_pos_l1"], "--", label="val (pos L1)")
    a.set_title(f"seed {s}: best epoch {mets['model_b'][str(s)]['best_epoch']}"); a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()
rows = []
for sp in ("train", "val"):
    for name, m in (("Model B (mean of seeds)", {k: v["mean"] for k, v in mets["model_b_mean_sd"][sp].items()}), ("B0", mets["B0"][sp]), ("B1", mets["B1"][sp])):
        rows.append({"split": sp, "method": name, **{k: round(m[k], 3) for k in ("mpjpe_body", "mpjpe_lhand", "mpjpe_rhand", "velocity_ratio", "diversity_ratio")}})
display(pd.DataFrame(rows))
print({s: (mets["model_b"][s]["best_epoch"], mets["model_b"][s]["epochs"], mets["train_seconds"][s]) for s in mets["model_b"]}, "(best epoch, epochs run, seconds)")

In [ ]:
S.finish(checks=[
    ("dry run ran on the GPU (cuda)", dry_meta["device"].startswith("cuda")),
    ("dry-run loss finite", all(d["finite"] for d in dry_meta["dry_run"].values())),
    ("3 seeds finished", len(mets["model_b"]) == 3),
    ("run is marked for the paper (clean code)", meta["for_paper"]),
    ("resume reproduces the weights (max |diff| < 1e-3)", max_diff < 1e-3),
])